# Creating Sentence Embeddings

Different from the seminal paper [Word2Vec](https://arxiv.org/abs/1301.3781) that introduced vectorized embeddings for **words** to do word similarity tasks, sentence embeddings deal with **sentences** and use the [transformer architecture](https://huggingface.co/sentence-transformers) to do so. All of these examples come from chapter 10 of the book [Hands on Large-Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961/ref=sr_1_1?dib=eyJ2IjoiMSJ9.WVja6YNL5PgDJvnmO9EzK1CvsXuvANIrEEeIFnZdfr_aCC1ksqWLqsh-Ggca5dAEbjCDmeumLwSQd0F-TAnmMWNvtPeD43E-Cpugm7m5sm5MxZ98F7pX-tmshG5SNClJ8h0OFiHBwnAgu9hP9MP3AZkATmllKoXqncIQvuLaIcylkA21WvastvgvzlZe70uahoINZd8sF_Fi5OSbyl3M747-6OClcVe43UkgJsaCNVQ.lB8wdaBNGsvPPTqBZmfIMVdaaGtfBJS2s4rxTN3c-l0&dib_tag=se&hvadid=693908941154&hvdev=c&hvexpln=67&hvlocphy=9006598&hvnetw=g&hvocijid=11837027820605730846--&hvqmt=e&hvrand=11837027820605730846&hvtargid=kwd-2098329602793&hydadcr=19108_13356960&keywords=hands+on+large+language+models&mcid=80c3f10598a0348682f5d44de4792183&qid=1742836934&sr=8-1)

## Training embedding Models from scratch

In [ ]:
# install packages
!pip install datasets # https://pypi.org/project/datasets/
!pip install sentence_transformers # https://pypi.org/project/sentence-transformers/

In [ ]:
# import modules
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers import losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader
from torch import Tensor, nn
import numpy as np
import pandas as pd
import random

In [ ]:
# downloading mnli and stsb dataset from the glue benchmrk: https://huggingface.co/datasets/nyu-mll/glue

# creating training dataset for Softmax loss
train_dataset = load_dataset(
    "glue", "mnli", split="train"
).select(range(50_000))
train_dataset = train_dataset.remove_columns(["idx"])

# preprocessing training dataset for MultipleNegativesRankingLoss
mnli = train_dataset.filter(lambda x: True if x["label"] == 0 else False)
train_dataset_two = {"anchor": [], "positive": [], "negative": []}
soft_negatives = mnli["hypothesis"]
random.shuffle(soft_negatives)
for row, soft_negative in zip(mnli, soft_negatives):
    train_dataset_two["anchor"].append(row["premise"])
    train_dataset_two["positive"].append(row["hypothesis"])
    train_dataset_two["negative"].append(soft_negative)
train_dataset_two = Dataset.from_dict(train_dataset_two)

# creating validation dataset
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

In [ ]:
# Downloading base models
embedding_model = SentenceTransformer('stsb-bert-base')
embedding_model_two = SentenceTransformer('microsoft/mpnet-base')

### SoftMaxLoss

In [ ]:
train_loss_bert = losses.SoftmaxLoss(model=embedding_model, sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(), num_labels=3)
train_loss_mpnet = losses.SoftmaxLoss(model=embedding_model_two, sentence_embedding_dimension=embedding_model_two.get_sentence_embedding_dimension(), num_labels=3)

In [ ]:
# bert

# defining args for bert
args_one = SentenceTransformerTrainingArguments(
    output_dir="bert_embedding_model_softmax",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args_one,
    train_dataset=train_dataset,
    loss=train_loss_bert,
    evaluator=evaluator
)
trainer.train()
evaluator(embedding_model)

In [ ]:
# Mpnet

# defining args for mpnet
args = SentenceTransformerTrainingArguments(
    output_dir="mpnet_embedding_model_softmax",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train mpnetembedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model_two,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss_mpnet,
    evaluator=evaluator
)
trainer.train()
evaluator(embedding_model_two)

### Multiple negatives ranking (MNR) loss

In [ ]:
train_loss_bert = losses.MultipleNegativesRankingLoss(model=embedding_model)
train_loss_mpnet = losses.MultipleNegativesRankingLoss(model=embedding_model_two)

In [ ]:
# bert

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model_bert",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset_two,
    loss=train_loss_bert,
    evaluator=evaluator
)
trainer.train()
evaluator(embedding_model)

In [ ]:
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model_mpnet",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)
# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model_two,
    args=args,
    train_dataset=train_dataset_two,
    loss=train_loss_mpnet,
    evaluator=evaluator
)
trainer.train()
evaluator(embedding_model_two)

## Fine-Tuning Pre-trained Models

### Supervised/Fine-Tuning with all-MiniLM-L6-v2

The pre-trained model [all-MiniLM-L6-v2](https://https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) is based on Microsoft's distilled model [MiniLM](https://huggingface.co/microsoft/MiniLM-L12-H384-uncased) which is a  12-layer encoder based Transformer model that is uncased and produces a 384 dimensional embedding. ALL-MiniLM-L6-v2 is a 6 layer [variant](https://huggingface.co/nreimers/MiniLM-L6-H384-uncased) of Microsoft's pre-trained model.

From a fine-tuning perspective all-MiniLM-L6-v2 was fine-tuned on 1B sentence pairs of the following datasets:

<div class="max-w-full overflow-auto">
<table>
<thead><tr>
<th>Dataset</th>
<th align="center">Paper</th>
<th align="center">Number of training tuples</th>
</tr>
</thead><tbody><tr>
<td><a rel="nofollow" href="https://github.com/PolyAI-LDN/conversational-datasets/tree/master/reddit">Reddit comments (2015-2018)</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/1904.06472">paper</a></td>
<td align="center">726,484,430</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/s2orc">S2ORC</a> Citation pairs (Abstracts)</td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/2020.acl-main.447/">paper</a></td>
<td align="center">116,288,806</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/afader/oqa#wikianswers-corpus">WikiAnswers</a> Duplicate question pairs</td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.1145/2623330.2623677">paper</a></td>
<td align="center">77,427,422</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/facebookresearch/PAQ">PAQ</a> (Question, Answer) pairs</td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/2102.07033">paper</a></td>
<td align="center">64,371,441</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/s2orc">S2ORC</a> Citation pairs (Titles)</td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/2020.acl-main.447/">paper</a></td>
<td align="center">52,603,982</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/s2orc">S2ORC</a> (Title, Abstract)</td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/2020.acl-main.447/">paper</a></td>
<td align="center">41,769,185</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> (Title, Body) pairs</td>
<td align="center">-</td>
<td align="center">25,316,456</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> (Title+Body, Answer) pairs</td>
<td align="center">-</td>
<td align="center">21,396,559</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> (Title, Answer) pairs</td>
<td align="center">-</td>
<td align="center">21,396,559</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://microsoft.github.io/msmarco/">MS MARCO</a> triplets</td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.1145/3404835.3462804">paper</a></td>
<td align="center">9,144,553</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/gooaq">GOOAQ: Open Question Answering with Diverse Answer Types</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/pdf/2104.08727.pdf">paper</a></td>
<td align="center">3,012,496</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://www.kaggle.com/soumikrakshit/yahoo-answers-dataset">Yahoo Answers</a> (Title, Answer)</td>
<td align="center"><a rel="nofollow" href="https://proceedings.neurips.cc/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html">paper</a></td>
<td align="center">1,198,260</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/code_search_net">Code Search</a></td>
<td align="center">-</td>
<td align="center">1,151,414</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://cocodataset.org/#home">COCO</a> Image captions</td>
<td align="center"><a rel="nofollow" href="https://link.springer.com/chapter/10.1007%2F978-3-319-10602-1_48">paper</a></td>
<td align="center">828,395</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/allenai/specter">SPECTER</a> citation triplets</td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.18653/v1/2020.acl-main.207">paper</a></td>
<td align="center">684,100</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://www.kaggle.com/soumikrakshit/yahoo-answers-dataset">Yahoo Answers</a> (Question, Answer)</td>
<td align="center"><a rel="nofollow" href="https://proceedings.neurips.cc/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html">paper</a></td>
<td align="center">681,164</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://www.kaggle.com/soumikrakshit/yahoo-answers-dataset">Yahoo Answers</a> (Title, Question)</td>
<td align="center"><a rel="nofollow" href="https://proceedings.neurips.cc/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html">paper</a></td>
<td align="center">659,896</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/search_qa">SearchQA</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/1704.05179">paper</a></td>
<td align="center">582,261</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/eli5">Eli5</a></td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.18653/v1/p19-1346">paper</a></td>
<td align="center">325,475</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://shannon.cs.illinois.edu/DenotationGraph/">Flickr 30k</a></td>
<td align="center"><a rel="nofollow" href="https://transacl.org/ojs/index.php/tacl/article/view/229/33">paper</a></td>
<td align="center">317,695</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> Duplicate questions (titles)</td>
<td align="center"></td>
<td align="center">304,525</td>
</tr>
<tr>
<td>AllNLI (<a rel="nofollow" href="https://nlp.stanford.edu/projects/snli/">SNLI</a> and <a rel="nofollow" href="https://cims.nyu.edu/~sbowman/multinli/">MultiNLI</a></td>
<td align="center"><a rel="nofollow" href="https://doi.org/10.18653/v1/d15-1075">paper SNLI</a>, <a rel="nofollow" href="https://doi.org/10.18653/v1/n18-1101">paper MultiNLI</a></td>
<td align="center">277,230</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> Duplicate questions (bodies)</td>
<td align="center"></td>
<td align="center">250,519</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/flax-sentence-embeddings/stackexchange_xml">Stack Exchange</a> Duplicate questions (titles+bodies)</td>
<td align="center"></td>
<td align="center">250,460</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/google-research-datasets/sentence-compression">Sentence Compression</a></td>
<td align="center"><a rel="nofollow" href="https://www.aclweb.org/anthology/D13-1155/">paper</a></td>
<td align="center">180,000</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/pvl/wikihow_pairs_dataset">Wikihow</a></td>
<td align="center"><a rel="nofollow" href="https://arxiv.org/abs/1810.09305">paper</a></td>
<td align="center">128,542</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://github.com/chridey/altlex/">Altlex</a></td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/P16-1135.pdf">paper</a></td>
<td align="center">112,696</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://quoradata.quora.com/First-Quora-Dataset-Release-Question-Pairs">Quora Question Triplets</a></td>
<td align="center">-</td>
<td align="center">103,663</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://cs.pomona.edu/~dkauchak/simplification/">Simple Wikipedia</a></td>
<td align="center"><a rel="nofollow" href="https://www.aclweb.org/anthology/P11-2117/">paper</a></td>
<td align="center">102,225</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://ai.google.com/research/NaturalQuestions">Natural Questions (NQ)</a></td>
<td align="center"><a rel="nofollow" href="https://transacl.org/ojs/index.php/tacl/article/view/1455">paper</a></td>
<td align="center">100,231</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://rajpurkar.github.io/SQuAD-explorer/">SQuAD2.0</a></td>
<td align="center"><a rel="nofollow" href="https://aclanthology.org/P18-2124.pdf">paper</a></td>
<td align="center">87,599</td>
</tr>
<tr>
<td><a rel="nofollow" href="https://huggingface.co/datasets/trivia_qa">TriviaQA</a></td>
<td align="center">-</td>
<td align="center">73,346</td>
</tr>
<tr>
<td><strong>Total</strong></td>
<td align="center"></td>
<td align="center"><strong>1,170,060,424</strong></td>
</tr>
</tbody>
</table>
</div>

fine-tuning was based using a contrastive learning objective, in which the cosine similarity from each possible sentence pair from the batch was computed and then the cross entropy loss was computed by comparing it with the true pair. As detailed by the [authors](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) the model is intended to be used as a sentence and short paragraph encoder. Given an input text, it outputs a vector which captures the semantic information. The sentence vector may be used for information retrieval, clustering or sentence similarity tasks.





In [ ]:
train_dataset_three = train_dataset_two
train_dataset_three

In [ ]:
# Define model
fine_embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=fine_embedding_model)
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=fine_embedding_model,
    args=args,
    train_dataset=train_dataset_three,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()
# Evaluate our trained model
evaluator(embedding_model)

### Augmented SBERT

As the authors of the book Hands-On Large Language Models detail Augmented SBERT is a useful technique when you don't have alot of labeled data (which for most of us commoners is a fact of life).

It uses the cross-encoder architecture as found in Bert to assign labels to unlabeled data (i.e. pairs of sentences). Then this newly labeled data is used for fine-tuning a bi-encoder (SBERT).  

The four steps to this method are the following:
1.   Fine-tune a cross-encoder (BERT) **using a small, annotated dataset** (called the gold dataset)
2.   Assign labels to unlabled data (i.e. sentence pairs) with the fine-turned cross-encoder (called the silver dataset)
3. Fine-tune a bi-encoder(SBERT) on both datasets (i.e. the gold + silver dataset)



In [ ]:
# using https://huggingface.co/datasets/BlackKakapo/RoSTSC for gold set
dataset = load_dataset("BlackKakapo/RoSTSC", split="train").select(range(10000))

In [ ]:
dataset

In [ ]:
# Convert Dataset type into a Dataloader type for cross-encoder
gold_examples = [InputExample(texts=[row['sentence1'], row['sentence2']], label=row['score']) for row in dataset]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)
# Pandas DataFrame for easier data handling
gold = pd.DataFrame({
    "sentence1": dataset['sentence1'],
    "sentence2": dataset['sentence2'],
    "label": dataset['score']
})

In [ ]:
# Step 1: training cross-encoder with gold set
model = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
cross_encoder = CrossEncoder(model)
cross_encoder.fit(
    train_dataloader=gold_dataloader,
    loss_fct=nn.CrossEntropyLoss(),
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

In [ ]:
# creating silver unlabeled dataset
'''
Note: this example is assuming the data is
unlabeled eventhough the data is labeled;
this is done by removing the labeling score; in real life
data should be unlabeled and not come from the same
dataset as the gold dataset
'''
silver = load_dataset("BlackKakapo/RoSTSC", split="train").select(range(20000, 70000))
pairs = list(zip(silver["sentence1"], silver["sentence2"]))

In [ ]:
# unlabeled sentence pair
pairs[0]

In [ ]:
# step 2: Labeling the unlabeled sentence pairs using our fine-tuned cross-encoder
output = cross_encoder.predict(
    pairs, apply_softmax=True,
show_progress_bar=True
)
silver = pd.DataFrame(
    {
        "sentence1": silver["sentence1"],
        "sentence2": silver["sentence2"],
        "label": np.argmax(output, axis=0)
    }
)

In [ ]:
# Combine gold + silver training dataset
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=["sentence1", "sentence2"], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

In [ ]:
# Step 3

# Defining embedding model: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
model = SentenceTransformer(model)
# Loss function
train_loss = losses.CosineSimilarityLoss(model=model)
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss
)
trainer.train()

In [ ]:
embeddings = model.encode(["Nu e o idee bună", "Nu este o idee bună", "Va fi noros toată ziua"])
similarities = model.similarity(embeddings, embeddings)
similarities